# HyperStreamDB: 15-Minute Live Demo (Part 2)
## Building a RAG Pipeline

In this notebook, we'll build a full **Retrieval-Augmented Generation (RAG)** pipeline using:
1.  **SQuAD Dataset** as our knowledge base.
2.  **HyperStreamDB** for real-time semantic retrieval.
3.  **Ollama (Llama 3)** for local text generation.

### 1. The RAG Architecture

1. **Ingest**: Knowledge base chunks → Embedded → HyperStreamDB.
2. **Retrieve**: User Query → Embedded → Search in HyperStreamDB → TOP-K contexts.
3. **Generate**: Query + Contexts → LLM Prompt → Final Answer.

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import hyperstreamdb as hdb
import pandas as pd
import numpy as np
import requests
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Initialize embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

### 2. Ingest Knowledge Base (Wikipedia / SQuAD)

We'll take unique context paragraphs from SQuAD.

In [ ]:
dataset = load_dataset("squad", split="train")
unique_contexts = pd.DataFrame(dataset)["context"].unique()[:300]
df = pd.DataFrame({"context": unique_contexts})
df["title"] = [dataset[i]["title"] for i in range(300)]  # Roughly map some titles

print(f"Embedding {len(df)} Wikipedia paragraphs...")
# Use numpy arrays directly - HyperStreamDB accepts them natively
embeddings = model.encode(df["context"].tolist())
print(f"✅ Embeddings format: {type(embeddings[0])}, Shape: {embeddings[0].shape}, Dtype: {embeddings[0].dtype}")
df["embedding"] = list(embeddings)  # Each row gets a numpy array directly

# Write to HyperStreamDB

table = hdb.Table("./rag_db")
table.add_index_columns(["embedding"])
table.write_pandas(df)
table.commit() # Create partition layout out of index definitions!
print("Knowledge base ingested with natively built semantic graph.")

### 3. Retrieval Function

Given a question, search HyperStreamDB for relevant passages.

In [ ]:
def retrieve(question, k=3, max_distance=None):
    """Retrieve relevant contexts with optional distance filtering"""
    q_emb = model.encode(question)  # Use numpy array directly
    print(f"Query embedding format: {type(q_emb)}, Shape: {q_emb.shape}")
    
    results = table.to_pandas(vector_filter={"column": "embedding", "query": q_emb, "k": k})

    if max_distance is not None:
        print(f"Applying distance filter: max_distance <= {max_distance}")
        results = results[results['distance'] <= max_distance]
    
    return results["context"].tolist()

# Test basic retrieval
question = "Who was the first person to win two Nobel Prizes?"
print(f"\n=== BASIC RETRIEVAL ===")
contexts = retrieve(question)
for i, ctx in enumerate(contexts):
    print(f"--- RELEVANT CONTEXT {i+1} ---\n{ctx[:200]}...\n")

# Test with distance filtering for high-quality results
print(f"\n=== HIGH-QUALITY RETRIEVAL (distance filter) ===")
quality_contexts = retrieve(question, k=5, max_distance=0.6)
print(f"Found {len(quality_contexts)} high-quality matches")

### 4. Generation with Ollama

We'll use a local Ollama instance.

In [ ]:
def generate_answer(question, contexts):
    prompt = f"""Answer the question accurately using only the provided context.

Context:
"""
    for ctx in contexts:
        prompt += f"- {ctx}\n"
    
    prompt += f"\nQuestion: {question}\nAnswer:"
    
    try:
        r = requests.post("http://localhost:11434/api/generate", 
                          json={"model": "llama3", "prompt": prompt, "stream": False}, 
                          timeout=10)
        return r.json()["response"]
    except:
        return "[Ollama not running. Returning retrieved context instead]\n\n" + "\n".join(contexts)

### 6. Final Live Demo: Enhanced RAG with Quality Control

### 5. Distance Filtering for Quality Control

HyperStreamDB supports distance-based filtering to ensure only high-quality, semantically relevant results are returned in your RAG pipeline.

In [ ]:
# Demonstrate distance filtering at different quality levels
test_query = "What are the effects of climate change?"

print("=== DISTANCE FILTERING COMPARISON ===\n")

# Very strict: only highly relevant results
print("1. STRICT FILTER (distance ≤ 0.4):")
strict_results = retrieve(test_query, k=5, max_distance=0.4)
print(f"   Found: {len(strict_results)} results\n")

# Moderate: balanced relevance
print("2. MODERATE FILTER (distance ≤ 0.7):")
moderate_results = retrieve(test_query, k=5, max_distance=0.7)
print(f"   Found: {len(moderate_results)} results\n")

# Permissive: broader context
print("3. PERMISSIVE FILTER (distance ≤ 1.0):")
permissive_results = retrieve(test_query, k=5, max_distance=1.0)
print(f"   Found: {len(permissive_results)} results\n")

# No filter: all top-k results
print("4. NO FILTER (top-k only):")
no_filter_results = retrieve(test_query, k=5)
print(f"   Found: {len(no_filter_results)} results\n")

print("📊 Quality vs Quantity: Stricter filters = Higher relevance, potentially fewer results")

In [ ]:
def rag(question, k=3, max_distance=0.7, use_quality_filter=True):
    """Enhanced RAG function with distance filtering for better quality control"""
    print(f"User Question: {question}")
    print("Searching knowledge base...")
    
    if use_quality_filter:
        print(f"Using quality filter: max_distance={max_distance}")
        contexts = retrieve(question, k=k, max_distance=max_distance)
    else:
        contexts = retrieve(question, k=k)
    
    print(f"Found {len(contexts)} relevant contexts")
    print("Generating answer...")
    answer = generate_answer(question, contexts)
    print(f"\nFINAL ANSWER:\n{answer}")
    return answer

# Demo 1: Standard RAG with quality filtering
print("=== RAG WITH QUALITY FILTER ===")
rag("In what year did Notre Dame's main building burn down?")

print("\n" + "="*60 + "\n")

# Demo 2: Compare with and without distance filtering
print("=== RAG WITHOUT QUALITY FILTER ===")
rag("What is machine learning?", use_quality_filter=False)

print("\n" + "="*60 + "\n")

# Demo 3: Strict quality filter for highly relevant results only
print("=== STRICT QUALITY FILTER (distance < 0.5) ===")
rag("Tell me about scientific discoveries", max_distance=0.5)